# Feature Engineering: Part 2

We have seen how to tweak columns to create more representative variables that can improve our models.<br><br>
In this section, we shall see how we can do feature engineering:
- sequentially using a pipeline
- on multiple columns at one go

## Deciding on which feature engineering method to use
- What is essential for my model (e.g. Logistic Regression)<br><br>
    - Categorical columns: impute + one-hot-encode<br><br>
    - Numerical columns: impute + scale (and Possibly bin and one-hot-encode)<br><br>
   
 - Some questions to ponder:<br><br>
     - Would it help if I bin the age?<br><br>
     - Would it help if I impute missing values/scale etc.<br><br>

## Import libraries from previous feature engineering lesson

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import SimpleImputer
pd.options.mode.chained_assignment = None 

# 1 Business goal
Train a logistic regression model that can predict Penguins belonging to either the Gentoo or the Adelie species


# 2 Get data

In [2]:
df = pd.read_csv('../3_Intro_to_FeatureEngineering/penguins_simple.csv', sep=';')

In [3]:
df.tail()

,Species,Culmen Length (mm),Culmen Depth (mm),Flipper Length (mm),Body Mass (g),Sex
328,Gentoo,47.2,13.7,214.0,4925.0,FEMALE
329,Gentoo,46.8,14.3,215.0,4850.0,FEMALE
330,Gentoo,50.4,15.7,222.0,5750.0,MALE
331,Gentoo,45.2,14.8,212.0,5200.0,FEMALE
332,Gentoo,49.9,16.1,213.0,5400.0,MALE


### Let's add a fictious date column

In [4]:
np.random.seed(42)
some_dates = pd.date_range(start="2015/06/14", end="2020/03/24")

some_dates

DatetimeIndex(['2015-06-14', '2015-06-15', '2015-06-16', '2015-06-17',
               '2015-06-18', '2015-06-19', '2015-06-20', '2015-06-21',
               '2015-06-22', '2015-06-23',
               ...
               '2020-03-15', '2020-03-16', '2020-03-17', '2020-03-18',
               '2020-03-19', '2020-03-20', '2020-03-21', '2020-03-22',
               '2020-03-23', '2020-03-24'],
              dtype='datetime64[ns]', length=1746, freq='D')

In [5]:
# Select a sample of dates from some_dates corresponding to the no of rows of df
df["date"] = np.random.choice(some_dates, size=len(df))
df.head()

,Species,Culmen Length (mm),Culmen Depth (mm),Flipper Length (mm),Body Mass (g),Sex,date
0,Adelie,39.1,18.7,181.0,3750.0,MALE,2018-07-14
1,Adelie,39.5,17.4,186.0,3800.0,FEMALE,2019-06-12
2,Adelie,40.3,18.0,195.0,3250.0,FEMALE,2017-10-21
3,Adelie,36.7,19.3,193.0,3450.0,FEMALE,2018-12-29
4,Adelie,39.3,20.6,190.0,3650.0,MALE,2018-07-18


# 3 Train-Test-Split
- We define X and y and then train-test-split

In [6]:
X = df.drop("Species", axis=1)
y = df["Species"]

# For the sake of this lesson we use the whole X as our X_train
X_train = X.copy()

# In your case, you must do:
from sklearn.model_selection import train_test_split

# X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.2, random_state = 42)

In [7]:
X_train.head()

,Culmen Length (mm),Culmen Depth (mm),Flipper Length (mm),Body Mass (g),Sex,date
0,39.1,18.7,181.0,3750.0,MALE,2018-07-14
1,39.5,17.4,186.0,3800.0,FEMALE,2019-06-12
2,40.3,18.0,195.0,3250.0,FEMALE,2017-10-21
3,36.7,19.3,193.0,3450.0,FEMALE,2018-12-29
4,39.3,20.6,190.0,3650.0,MALE,2018-07-18


# 4 Explore the data

# 5 Feature engineering 

## Introducing `FunctionTransformer`
Sometimes, the tranformer function we want is not available in `sklearn`.<br><br>
We can create a custom function and wrap it with a `FunctionTransformer` from `sklearn` then apply the `fit` and `transform` methods<br><br>
The function we write must accept a dataframe among other arguments<br><br>
Let's import `FunctionTransformer` from `sklearn`

In [8]:
from sklearn.preprocessing import FunctionTransformer

- Create a function that extracts the month in date column

In [9]:
def extract_month(my_df, column):
    return pd.DataFrame(my_df[column].dt.month)

In [10]:
# Test first to see that it works
extract_month(X_train, "date")

,date
0,7
1,6
2,10
3,12
4,7
...,...
328,11
329,7
330,1
331,9


- Wrap the function inside `FunctionTransformer`

In [11]:
MyDateTransformer = FunctionTransformer(extract_month, kw_args={"column": "date"})

- Fit and transform the column

In [12]:
MyDateTransformer.fit_transform(X_train)

,date
0,7
1,6
2,10
3,12
4,7
...,...
328,11
329,7
330,1
331,9


## Introducing Pipeline

We can perform feature engineering on a column in multiple sequential steps.<br><br>
We pass a list of sklearn tranformers for preprocessing<br><br>
The `sklearn` variant is `Pipeline` and `make_pipeline`. <br><br>
We shall use `make_pipeline`

In [13]:
df.isna().sum()

Species                0
Culmen Length (mm)     0
Culmen Depth (mm)      0
Flipper Length (mm)    0
Body Mass (g)          0
Sex                    0
date                   0
dtype: int64

In [14]:
from sklearn.pipeline import make_pipeline

### We can perform the following steps in a sequential order using `make_pipeline`
- impute missing data on categorical variables by the most frequent observation (mode)<br><br>
- one-hot-encode categorical variables (Sex) and ordinal variables (month)<br><br>
- scale numerical variables in the range [0, 1]<br><br>
- bin numerical variables into 5 categories

### Introduce nans to the Sex column
- To demonstrate imputation, we shall introduce `nans` randomly to the Sex column<br><br>
- Assign Nan to all values between index 2 and index 50 at intervals of 10 for the Sex column

In [15]:
X_train["Sex"][2:50:10] = np.nan
# X_train["Sex"].isna().sum()

### Categorical pipe
- impute and one-hot-encode

In [16]:
categorical_pipe = make_pipeline(
    SimpleImputer(strategy="most_frequent"), # First step
    OneHotEncoder(sparse=False, drop="first") # Second step
        
)

- **Apply the pipe to the Sex variable**

In [17]:
categorical_pipe.fit(df[["Sex"]])

/media/joseph/spiced/cluster-royco-encounter-notes/venv/lib/python3.8/site-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


Pipeline(steps=[('simpleimputer', SimpleImputer(strategy='most_frequent')),
                ('onehotencoder',
                 OneHotEncoder(drop='first', sparse=False,
                               sparse_output=False))])

- **What features are there in the categorical pipe?**

In [18]:
categorical_pipe.get_feature_names_out()

array(['Sex_MALE'], dtype=object)

**Transform the Sex column**

In [19]:
categorical_pipe.transform(df[["Sex"]])

array([[1.],
       [0.],
       [0.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [1.],
       [1.],
       [0.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [1.],
       [1.],
       [0.],
       [1.],
       [0.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [1.],
       [1.],
       [0.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [1.],
       [1.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.],

### Extract the month column and one-hot-encode
We want to use our date transformer inside a pipeline. <br><br>
- Steps:<br><br>
    - Extract month from date<br><br>
    - One-Hot-Encode the month<br><br>

In [20]:
extract_and_ohe_pipe = make_pipeline(
    MyDateTransformer, # First step
    OneHotEncoder(drop="first", sparse=False) # second step
)

- `fit_transform` the pipe to the date column

In [21]:
extract_and_ohe_pipe.fit_transform(X_train)

/media/joseph/spiced/cluster-royco-encounter-notes/venv/lib/python3.8/site-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 1., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

### Numerical pipe
For the numerical pipeline we need to:
- impute missing values<br><br>
- squish the values to within a range of 0 and 1 <br><br>
- discretize the result into 5 bins<br><br>


In [22]:
numerical_pipe = make_pipeline(
    SimpleImputer(strategy="mean"),
    MinMaxScaler(),
    KBinsDiscretizer(n_bins=5, strategy="quantile", encode="onehot-dense")
)

- Isolate the numerical columns

In [23]:
numerical_df = X_train.drop(["Sex", "date"], axis=1)

- Fit the pipe

In [24]:
numerical_pipe.fit(numerical_df)

Pipeline(steps=[('simpleimputer', SimpleImputer()),
                ('minmaxscaler', MinMaxScaler()),
                ('kbinsdiscretizer', KBinsDiscretizer(encode='onehot-dense'))])

- Transform numerical columns

In [25]:
numerical_pipe.transform(numerical_df)

array([[0., 1., 0., ..., 0., 0., 0.],
       [0., 1., 0., ..., 1., 0., 0.],
       [0., 1., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 1.],
       [0., 0., 1., ..., 0., 0., 1.],
       [0., 0., 0., ..., 0., 0., 1.]])

## Introducing Column Transformer

- We can perform feature engineering on variables by defining which estimators (single ones or pipelines) you want to apply to each column in your dataframe

In [26]:
from sklearn.compose import ColumnTransformer

### Introduce nans to all but species columns

In [27]:
np.random.seed(42)
for column in X_train.columns: # Go through the columns one by one
    if column not in ["date"]: # Do not consider the date column
        series = list(X_train[column]) # grab a series for this column and convert to list
        indices = np.random.choice(range(len(X_train)), size=np.random.randint(1, 15)) # Get some random indices
        for i in indices: # Go through the indices
            series[i] = np.nan # Assign each value corresponding to an index a Nan value
        X_train[column] = series  # Update the column with a few values assigned nan

In [28]:
X_train.isna().sum()

Culmen Length (mm)      7
Culmen Depth (mm)       3
Flipper Length (mm)     5
Body Mass (g)           2
Sex                    17
date                    0
dtype: int64

In [29]:
X_train

,Culmen Length (mm),Culmen Depth (mm),Flipper Length (mm),Body Mass (g),Sex,date
0,39.1,18.7,181.0,3750.0,MALE,2018-07-14
1,39.5,17.4,186.0,3800.0,FEMALE,2019-06-12
2,40.3,18.0,195.0,3250.0,NaN,2017-10-21
3,36.7,19.3,193.0,3450.0,FEMALE,2018-12-29
4,39.3,20.6,190.0,3650.0,MALE,2018-07-18
...,...,...,...,...,...,...
328,47.2,13.7,214.0,4925.0,FEMALE,2018-11-21
329,46.8,14.3,215.0,4850.0,FEMALE,2018-07-10
330,50.4,NaN,222.0,5750.0,MALE,2020-01-21
331,45.2,14.8,212.0,5200.0,FEMALE,2017-09-06


### Reset index
- We want to have one column which we don't want to transform at all and watch how `ColumnTransformer` handles it

In [30]:
X_train_reset_index = X_train.reset_index()

In [31]:
X_train_reset_index

,index,Culmen Length (mm),Culmen Depth (mm),Flipper Length (mm),Body Mass (g),Sex,date
0,0,39.1,18.7,181.0,3750.0,MALE,2018-07-14
1,1,39.5,17.4,186.0,3800.0,FEMALE,2019-06-12
2,2,40.3,18.0,195.0,3250.0,NaN,2017-10-21
3,3,36.7,19.3,193.0,3450.0,FEMALE,2018-12-29
4,4,39.3,20.6,190.0,3650.0,MALE,2018-07-18
...,...,...,...,...,...,...,...
328,328,47.2,13.7,214.0,4925.0,FEMALE,2018-11-21
329,329,46.8,14.3,215.0,4850.0,FEMALE,2018-07-10
330,330,50.4,NaN,222.0,5750.0,MALE,2020-01-21
331,331,45.2,14.8,212.0,5200.0,FEMALE,2017-09-06


**With columntransformer, we can perform all the steps before at once and leave out the columns we don't need to feature-engineer**<br><br>
`sklearn.compose.ColumnTransformer` takes a list of transformers with a (`name`, `method`, `columns`) tuple.

- These are the transformations we need for each column:<br><br>

    - extract the month from date and one-hot-encode <br><br>
    - impute and one-hot-encode the Sex column <br><br>
    - impute, scale and discretize (bin) all numerical columns<br><br>
    - do nothing on the index column <br><br>
    - drop the rest of the columns (in this case the date column)<br><br><br>
 

- Our `columntransformer` takes the following format:

In [50]:
feature_transform = ColumnTransformer(
    [
        ("extract_month_ohe", extract_and_ohe_pipe, ["date"]),
        ("impute_ohe", categorical_pipe, ["Sex"]),
        ("impute_scale_bin", numerical_pipe, list(numerical_df.columns)),
        ("do_nothing", "passthrough", ["index"])
        
    ]


)

In [52]:
extract_and_ohe_pipe

Pipeline(steps=[('functiontransformer',
                 FunctionTransformer(func=<function extract_month at 0x7f91bd5c10d0>,
                                     kw_args={'column': 'date'})),
                ('onehotencoder',
                 OneHotEncoder(drop='first', sparse=False,
                               sparse_output=False))])

In [51]:
list(numerical_df.columns)

['Culmen Length (mm)',
 'Culmen Depth (mm)',
 'Flipper Length (mm)',
 'Body Mass (g)']

### What type is the `feature_transform`?

In [44]:
type(feature_transform)

sklearn.compose._column_transformer.ColumnTransformer

### Fit the transformer with our `df_new` (in your case, `Xtrain`)

In [45]:
feature_transform.fit(X_train_reset_index)

/media/joseph/spiced/cluster-royco-encounter-notes/venv/lib/python3.8/site-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


ColumnTransformer(transformers=[('impute_ohe',
                                 Pipeline(steps=[('simpleimputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehotencoder',
                                                  OneHotEncoder(drop='first',
                                                                sparse=False,
                                                                sparse_output=False))]),
                                 ['Sex']),
                                ('impute_scale_bin',
                                 Pipeline(steps=[('simpleimputer',
                                                  SimpleImputer()),
                                                 ('minmaxscaler',
                                                  MinMaxScaler()),
                                                 ('kbinsdiscretizer',
                                                  KBinsDiscretizer(encode='onehot-dense'))]),
                                 ['Culmen Length (mm)', 'Culmen Depth (mm)',
                                  'Flipper Length (mm)', 'Body Mass (g)']),
                                ('do_nothing', 'passthrough', ['index'])])

In [37]:
X_train_reset_index

,index,Culmen Length (mm),Culmen Depth (mm),Flipper Length (mm),Body Mass (g),Sex,date
0,0,39.1,18.7,181.0,3750.0,MALE,2018-07-14
1,1,39.5,17.4,186.0,3800.0,FEMALE,2019-06-12
2,2,40.3,18.0,195.0,3250.0,NaN,2017-10-21
3,3,36.7,19.3,193.0,3450.0,FEMALE,2018-12-29
4,4,39.3,20.6,190.0,3650.0,MALE,2018-07-18
...,...,...,...,...,...,...,...
328,328,47.2,13.7,214.0,4925.0,FEMALE,2018-11-21
329,329,46.8,14.3,215.0,4850.0,FEMALE,2018-07-10
330,330,50.4,NaN,222.0,5750.0,MALE,2020-01-21
331,331,45.2,14.8,212.0,5200.0,FEMALE,2017-09-06


### Transform our data with the column transformer

In [46]:
X_train_fe = feature_transform.transform(X_train_reset_index)

In [47]:
X_train_fe.shape

(333, 22)

### in case you want to see the features after column transformation
- Remember that you would run into headwinds in case you tried to get the feature names out with a custom function tranformer.

In [48]:
feature_transform.get_feature_names_out() # won't work not unless you remove pipe with custom function

array(['impute_ohe__Sex_MALE', 'impute_scale_bin__Culmen Length (mm)_0.0',
       'impute_scale_bin__Culmen Length (mm)_1.0',
       'impute_scale_bin__Culmen Length (mm)_2.0',
       'impute_scale_bin__Culmen Length (mm)_3.0',
       'impute_scale_bin__Culmen Length (mm)_4.0',
       'impute_scale_bin__Culmen Depth (mm)_0.0',
       'impute_scale_bin__Culmen Depth (mm)_1.0',
       'impute_scale_bin__Culmen Depth (mm)_2.0',
       'impute_scale_bin__Culmen Depth (mm)_3.0',
       'impute_scale_bin__Culmen Depth (mm)_4.0',
       'impute_scale_bin__Flipper Length (mm)_0.0',
       'impute_scale_bin__Flipper Length (mm)_1.0',
       'impute_scale_bin__Flipper Length (mm)_2.0',
       'impute_scale_bin__Flipper Length (mm)_3.0',
       'impute_scale_bin__Flipper Length (mm)_4.0',
       'impute_scale_bin__Body Mass (g)_0.0',
       'impute_scale_bin__Body Mass (g)_1.0',
       'impute_scale_bin__Body Mass (g)_2.0',
       'impute_scale_bin__Body Mass (g)_3.0',
       'impute_scale_bin__B

- `X_train_fe` contains the feature-engineered features that we use to train our `LogisticRegression` model

# 6 Train model

# 7 Optimize

# 8 Evaluate model

# 9 Make Predictions